# AADS-ULoRA VLM Router (One-Click Colab)
Run cells in order. This notebook is standalone and GitHub/Colab ready.

In [ ]:
# Cell 1: Setup + create pipeline
import os, sys, json, subprocess
from pathlib import Path
from google.colab import userdata, drive

TOKEN_FILE_OVERRIDE = '/content/drive/MyDrive/huggingfacetoken.txt'
INSTALL_ULTRALYTICS_COMPAT = False  # Set True only if you want SAM2 fallback compatibility

subprocess.run('git clone https://github.com/EfeErim/bitirmeprojesi.git /content/bitirmeprojesi 2>/dev/null || (cd /content/bitirmeprojesi && git pull)', shell=True, check=False)
os.chdir('/content/bitirmeprojesi')

deps = [
    'pip install --upgrade pip',
    'pip install -q transformers>=4.41.0',
    'pip install -q open-clip-torch',
    'pip install -q huggingface-hub'
]
if INSTALL_ULTRALYTICS_COMPAT:
    deps.append('pip install -q ultralytics')

for cmd in deps:
    subprocess.run(cmd, shell=True, check=False)

drive.mount('/content/drive', force_remount=False)

token = None
try:
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None

if not token:
    p = Path(TOKEN_FILE_OVERRIDE)
    if p.exists() and p.is_file():
        token = p.read_text(encoding='utf-8').strip()

if not token:
    raise RuntimeError('HF_TOKEN not found. Set Colab Secret HF_TOKEN or TOKEN_FILE_OVERRIDE.')

os.environ['HF_TOKEN'] = token

sys.path.insert(0, '/content/bitirmeprojesi')
sys.path.insert(0, '/content/bitirmeprojesi/src')

for k in list(sys.modules.keys()):
    if k.startswith('src.router.vlm_pipeline') or k.startswith('router.vlm_pipeline'):
        del sys.modules[k]

try:
    from src.router.vlm_pipeline import VLMPipeline
except Exception:
    from router.vlm_pipeline import VLMPipeline

config = json.load(open('/content/bitirmeprojesi/config/colab.json', 'r', encoding='utf-8'))
pipeline = VLMPipeline(config)
print('✅ Setup complete, pipeline created')

In [ ]:
# Cell 2: Load models + verify
import torch

if 'pipeline' not in globals():
    raise RuntimeError('pipeline not found. Run Cell 1 first.')

already_loaded = bool(getattr(pipeline, 'models_loaded', False)) and (getattr(pipeline, 'actual_pipeline', None) == 'sam3')
if already_loaded:
    print('✅ Models already loaded (SAM3).')
else:
    print('⏳ Loading models...')
    pipeline.load_models()

sam3_loaded = getattr(pipeline, 'sam_model', None) is not None and getattr(pipeline, 'sam_backend', None) == 'sam3'
bioclip_loaded = getattr(pipeline, 'bioclip', None) is not None
backend = getattr(pipeline, 'bioclip_backend', 'unknown')
print(f'SAM3: {sam3_loaded} | BioCLIP: {bioclip_loaded} | backend={backend}')
if torch.cuda.is_available():
    print(f'GPU mem allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# Cell 3: Analyze uploaded image (all crops, interactive fast mode)
from google.colab import files
from PIL import Image
import json

taxonomy = json.load(open('/content/bitirmeprojesi/config/plant_taxonomy.json', 'r', encoding='utf-8'))
all_crops = taxonomy.get('crops', [])
pipeline.crop_labels = all_crops
pipeline.open_set_enabled = True
pipeline.open_set_min_confidence = 0.35
pipeline.open_set_margin = 0.03

# Fast interactive settings to prevent long "stuck analyzing" runs
FAST_MODE = True
if FAST_MODE:
    pipeline.vlm_config['sam3_mask_threshold'] = 0.08
    pipeline.vlm_config['min_box_area_ratio'] = 0.004
    pipeline.vlm_config['min_box_side_px'] = 18
    pipeline.vlm_config['classification_min_confidence'] = 0.40
    pipeline.vlm_config['detection_nms_iou_threshold'] = 0.60
    pipeline.vlm_config['conditioned_part_weight'] = 0.60
    pipeline.vlm_config['max_rois_for_classification'] = 4
    pipeline.vlm_config['generic_part_penalty'] = 0.65
    pipeline.vlm_config['generic_part_labels'] = ['whole plant', 'whole', 'plant', 'entire plant']
    pipeline.vlm_config['specific_part_override_ratio'] = 0.42
    pipeline.vlm_config['specific_part_min_confidence'] = 0.10
    pipeline.vlm_config['preferred_part_labels'] = ['leaf']
    pipeline.vlm_config['preferred_part_override_ratio'] = 0.45
    pipeline.vlm_config['leaf_override_enabled'] = True
    pipeline.vlm_config['leaf_override_label'] = 'leaf'
    pipeline.vlm_config['leaf_override_target_labels'] = ['whole plant', 'whole', 'plant', 'entire plant', 'fruit', 'berry']
    pipeline.vlm_config['leaf_override_ratio'] = 0.32
    pipeline.vlm_config['leaf_override_min_confidence'] = 0.08
    pipeline.vlm_config['leaf_override_min_area_ratio'] = 0.015
    pipeline.vlm_config['leaf_override_aspect_min'] = 0.25
    pipeline.vlm_config['leaf_override_aspect_max'] = 3.8
    if isinstance(pipeline.vlm_config.get('ensemble_config'), dict):
        pipeline.vlm_config['ensemble_config']['crop_num_prompts'] = 1
        pipeline.vlm_config['ensemble_config']['part_num_prompts'] = 1

MAX_DETECTIONS = 3 if FAST_MODE else 10
MAX_IMAGE_SIDE = 1024 if FAST_MODE else 1600

print(f'🔬 Full taxonomy enabled: {len(all_crops)} crops')
print(f'Thresholds: min_conf={pipeline.open_set_min_confidence}, margin={pipeline.open_set_margin}')
print(f"⚡ Fast mode: {FAST_MODE} | max_detections={MAX_DETECTIONS} | max_rois={pipeline.vlm_config.get('max_rois_for_classification')}")

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No image selected')

image_path = next(iter(uploaded.keys()))
pil_image = Image.open(image_path).convert('RGB')
orig_w, orig_h = pil_image.size
pil_image.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE), Image.Resampling.LANCZOS)
new_w, new_h = pil_image.size

print(f'🌱 Analyzing {image_path}...')
if (orig_w, orig_h) != (new_w, new_h):
    print(f'🖼️ Resized for speed: {orig_w}x{orig_h} -> {new_w}x{new_h}')

result = pipeline.analyze_image(pil_image, confidence_threshold=0.40, max_detections=MAX_DETECTIONS)
detections = result.get('detections', [])

print(f'\n✅ Done in {result.get("processing_time_ms", 0):.0f} ms')
print(f'📊 Detections found: {len(detections)}')
if result.get('stage_timings_ms'):
    print('⏱️ Stage timings (ms):', result.get('stage_timings_ms'))
if result.get('roi_stats'):
    print('🧩 ROI stats:', result.get('roi_stats'))

for i, det in enumerate(detections[:15], 1):
    crop = det.get('crop', 'unknown')
    crop_conf = det.get('crop_confidence', 0)
    part = det.get('part', 'unknown')
    part_conf = det.get('part_confidence', 0)
    print(f'{i:2d}. CROP={crop:15s} ({crop_conf:6.1%}) | PART={part:8s} ({part_conf:6.1%})')

In [ ]:
# Cell 4: Open-set diagnostics (explicit rejection reasons)
import torch

if 'detections' not in globals() or not detections:
    print('No detections. Run Cell 3 first.')
else:
    det = detections[0]
    bbox = det.get('bbox')
    roi = pipeline._extract_roi(pipeline._coerce_image_input(image_path)[0], bbox)

    crop_prompts, crop_map = pipeline._build_prompt_batch(pipeline.crop_labels, label_type='crop')
    unknown_prompts = pipeline._open_set_unknown_prompts(label_type='crop', known_labels=pipeline.crop_labels)

    if pipeline.bioclip_backend != 'open_clip':
        print(f'Backend is {pipeline.bioclip_backend}; expected open_clip for this diagnostic cell.')
    else:
        preprocess = pipeline.bioclip_processor['preprocess']
        tokenizer = pipeline.bioclip_processor['tokenizer']
        logit_scale = pipeline._get_clip_logit_scale(pipeline.bioclip)

        image_tensor = preprocess(roi).unsqueeze(0).to(pipeline.device)
        all_text = crop_prompts + unknown_prompts
        text_tokens = tokenizer(all_text).to(pipeline.device)

        with torch.no_grad():
            image_embeds = pipeline.bioclip.encode_image(image_tensor)
            text_embeds = pipeline.bioclip.encode_text(text_tokens)
            image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
            text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)
            prompt_logits = (image_embeds @ text_embeds.T) * logit_scale

        known_count = len(pipeline.crop_labels)
        prompt_to_class = crop_map + ([known_count] * len(unknown_prompts))
        class_count = known_count + 1
        agg_logits = pipeline._aggregate_prompt_logits(prompt_logits, prompt_to_class, class_count)
        probs = torch.softmax(agg_logits, dim=-1)[0].detach().cpu().numpy()

        crop_probs = probs[:known_count]
        unknown_prob = float(probs[known_count])
        top_idx = crop_probs.argsort()[::-1][:10]

        print('Top-10 crop probabilities:')
        for rank, idx in enumerate(top_idx, 1):
            print(f'  {rank:2d}. {pipeline.crop_labels[idx]:15s} {float(crop_probs[idx]):7.1%}')

        best_idx = int(top_idx[0])
        second_idx = int(top_idx[1]) if len(top_idx) > 1 else best_idx
        best_conf = float(crop_probs[best_idx])
        second_conf = float(crop_probs[second_idx])
        margin = best_conf - second_conf

        print('\nOpen-set decision terms:')
        print(f'  best crop:    {pipeline.crop_labels[best_idx]} ({best_conf:.1%})')
        print(f'  unknown prob: {unknown_prob:.1%}')
        print(f'  margin:       {margin:.4f}')
        print(f'  min_conf:     {pipeline.open_set_min_confidence:.2f}')
        print(f'  margin_th:    {pipeline.open_set_margin:.2f}')

        print('\nChecks:')
        print(f'  unknown>=best : {unknown_prob >= best_conf}')
        print(f'  best<min_conf : {best_conf < pipeline.open_set_min_confidence}')
        print(f'  margin<thresh : {margin < pipeline.open_set_margin}')

In [ ]:
# Cell 5: Summary table
import pandas as pd

if 'detections' not in globals() or not detections:
    print('No detections. Run Cell 3 first.')
else:
    rows = []
    for i, det in enumerate(detections, 1):
        rows.append({
            'ID': i,
            'Crop': det.get('crop', 'unknown'),
            'Crop Conf': f"{det.get('crop_confidence', 0):.1%}",
            'Part': det.get('part', 'unknown'),
            'Part Conf': f"{det.get('part_confidence', 0):.1%}",
            'BBox': det.get('bbox', 'N/A'),
        })
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    print('\nExport: df.to_csv("/content/detections.csv", index=False)')